# VoltShare Pricing ML Workflow

这个 notebook 对应当前小程序里真正用到的机器学习流程，不是单独写的一版 proposal 伪代码。它覆盖四件事：

1. 把 VIC1 `price + demand` 原始数据聚合成小时级 demand 序列  
2. 用前向时间切分训练 `OLS baseline` 和 `Random Forest`  
3. 从验证集残差中提取 demand-side 不确定性  
4. 把小时级 solar supply 数据接入 grid search 定价

Notebook 里的实现尽量贴近前端当前逻辑：

- demand 聚合规则参考 `scripts/generate-demand-side.mjs`
- supply 聚合规则参考 `scripts/generate-supply-side.mjs`
- 定价逻辑参考 `src/services/aiService.ts`

In [1]:
import importlib
import subprocess
import sys

REQUIRED = [
    ("scikit-learn", "sklearn"),
]

for package_name, module_name in REQUIRED:
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

ROOT = Path.cwd()
DEMAND_CSV = Path("/Users/jasmine/Desktop/project/PRICE_AND_DEMAND_202501_202603_VIC1_merged.csv")
SUPPLY_CSV = Path("/Users/jasmine/Desktop/Solar_Energy_Generation.csv")

HOUSEHOLD_SCALE_FACTOR = 4
TARGET_SURPLUS_KWH = 4.2
TARGET_PRICE = 0.175
LISTING_TIME = "2026-03-31T18:00:00Z"


def clamp(value, lower, upper):
    return min(upper, max(lower, value))


def percentile(values, probability):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 0.0
    return float(np.quantile(values, probability))


def compress_samples(values, max_count):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size <= max_count:
        return values

    indices = np.linspace(0, values.size - 1, max_count).astype(int)
    return values[indices]


def circular_distance(left, right, cycle):
    distance = abs(left - right)
    return min(distance, cycle - distance)


def metrics_row(name, actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return {
        "model": name,
        "rmse": mean_squared_error(actual, predicted, squared=False),
        "mae": mean_absolute_error(actual, predicted),
        "r2": r2_score(actual, predicted),
    }


def ensure_utc(timestamp_like):
    ts = pd.Timestamp(timestamp_like)
    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    else:
        ts = ts.tz_convert("UTC")
    return ts

ModuleNotFoundError: No module named 'matplotlib'

## 1. Demand side: 从原始 VIC1 数据构造小时级训练集

这里复现前端 demand 默认库的聚合规则：

- `TOTALDEMAND` 按小时取均值，再 `/ 1000`
- `RRP > 0` 的价格按小时取 `p90`
- 从 `$ / MWh` 转成 `$ / kWh`
- 再把接受价裁到 `[0.02, 0.4]`

In [ ]:
demand_raw = pd.read_csv(DEMAND_CSV)

required_demand_columns = {"REGION", "SETTLEMENTDATE", "TOTALDEMAND", "RRP"}
missing_demand_columns = required_demand_columns - set(demand_raw.columns)
if missing_demand_columns:
    raise ValueError(f"Demand CSV is missing columns: {sorted(missing_demand_columns)}")

demand_raw["SETTLEMENTDATE"] = pd.to_datetime(demand_raw["SETTLEMENTDATE"], utc=True, errors="coerce")
demand_raw["TOTALDEMAND"] = pd.to_numeric(demand_raw["TOTALDEMAND"], errors="coerce")
demand_raw["RRP"] = pd.to_numeric(demand_raw["RRP"], errors="coerce")
demand_raw = demand_raw.dropna(subset=["SETTLEMENTDATE", "TOTALDEMAND", "RRP"]).copy()

region = demand_raw["REGION"].dropna().iloc[0]


def aggregate_demand_hour(group):
    positive_prices = group.loc[group["RRP"] > 0, "RRP"].to_numpy()
    inferred_price = percentile(positive_prices, 0.9) / 1000 if positive_prices.size else 0.02
    return pd.Series(
        {
            "demandKwh": round(group["TOTALDEMAND"].mean() / 1000, 2),
            "maxPricePerKwh": round(clamp(inferred_price, 0.02, 0.4), 3),
        }
    )


hourly_demand = (
    demand_raw.assign(hour_bucket=demand_raw["SETTLEMENTDATE"].dt.floor("h"))
    .groupby("hour_bucket", sort=True)
    .apply(aggregate_demand_hour)
    .reset_index()
    .sort_values("hour_bucket")
    .reset_index(drop=True)
)

hourly_demand["buyerName"] = region + " demand " + hourly_demand["hour_bucket"].dt.strftime("%Y-%m-%d %H")
hourly_demand["requestedAt"] = hourly_demand["hour_bucket"]

print(f"Demand rows: {len(hourly_demand):,}")
print("Date range:", hourly_demand["hour_bucket"].min(), "to", hourly_demand["hour_bucket"].max())
display(hourly_demand.head())

In [ ]:
demand_features = hourly_demand.copy()
demand_features["hour"] = demand_features["hour_bucket"].dt.hour
demand_features["month"] = demand_features["hour_bucket"].dt.month
demand_features["dayOfWeek"] = demand_features["hour_bucket"].dt.dayofweek
demand_features["isWeekend"] = demand_features["dayOfWeek"].isin([5, 6]).astype(int)
demand_features["hourSin"] = np.sin(2 * np.pi * demand_features["hour"] / 24)
demand_features["hourCos"] = np.cos(2 * np.pi * demand_features["hour"] / 24)
demand_features["monthSin"] = np.sin(2 * np.pi * demand_features["month"] / 12)
demand_features["monthCos"] = np.cos(2 * np.pi * demand_features["month"] / 12)

demand_features["lagDemand1"] = demand_features["demandKwh"].shift(1)
demand_features["lagDemand24"] = demand_features["demandKwh"].shift(24)
demand_features["lagPrice1"] = demand_features["maxPricePerKwh"].shift(1)
demand_features["lagPrice24"] = demand_features["maxPricePerKwh"].shift(24)
demand_features["rollingDemand24"] = demand_features["demandKwh"].shift(1).rolling(24, min_periods=1).mean()
demand_features["rollingPrice24"] = demand_features["maxPricePerKwh"].shift(1).rolling(24, min_periods=1).mean()

lag_columns = [
    "lagDemand1",
    "lagDemand24",
    "lagPrice1",
    "lagPrice24",
    "rollingDemand24",
    "rollingPrice24",
]
demand_features[lag_columns] = demand_features[lag_columns].bfill().ffill()

demand_features["rollingDemand24"] = demand_features["rollingDemand24"].fillna(demand_features["demandKwh"])
demand_features["rollingPrice24"] = demand_features["rollingPrice24"].fillna(demand_features["maxPricePerKwh"])
demand_features["logPrice"] = np.log(demand_features["maxPricePerKwh"].clip(lower=0.005))
demand_features["logDemand"] = np.log(demand_features["demandKwh"].clip(lower=0.01))

row_count = len(demand_features)
train_end = max(72, int(row_count * 0.7))
validation_end = max(train_end + 24, int(row_count * 0.8))
validation_end = min(validation_end, row_count - 1)

train_df = demand_features.iloc[:train_end].copy()
validation_df = demand_features.iloc[train_end:validation_end].copy()
test_df = demand_features.iloc[validation_end:].copy()

print("Train rows:", len(train_df), train_df["hour_bucket"].min(), "to", train_df["hour_bucket"].max())
print("Validation rows:", len(validation_df), validation_df["hour_bucket"].min(), "to", validation_df["hour_bucket"].max())
print("Test rows:", len(test_df), test_df["hour_bucket"].min(), "to", test_df["hour_bucket"].max())

display(demand_features.head())

## 2. OLS baseline 与 Random Forest

这里对应前端的两套模型：

- `OLS baseline`: 预测 `log(demand)`  
- `Random Forest`: 直接预测 `demandKwh`

训练方式使用前向时间切分，不做随机打乱。

In [ ]:
def make_baseline_design(frame):
    base = frame[["logPrice", "isWeekend", "lagDemand1", "lagDemand24", "lagPrice1", "lagPrice24"]].reset_index(drop=True)
    hour_dummies = pd.get_dummies(frame["hour"], prefix="hour", drop_first=True, dtype=float).reset_index(drop=True)
    month_dummies = pd.get_dummies(frame["month"], prefix="month", drop_first=True, dtype=float).reset_index(drop=True)
    design = pd.concat([base, hour_dummies, month_dummies], axis=1)
    return sm.add_constant(design, has_constant="add")


baseline_train_X = make_baseline_design(train_df)
baseline_columns = baseline_train_X.columns.tolist()
baseline_validation_X = make_baseline_design(validation_df).reindex(columns=baseline_columns, fill_value=0.0)
baseline_test_X = make_baseline_design(test_df).reindex(columns=baseline_columns, fill_value=0.0)

ols_model = sm.OLS(train_df["logDemand"], baseline_train_X).fit()

baseline_validation_pred = np.clip(np.exp(ols_model.predict(baseline_validation_X).to_numpy()), 0, None)
baseline_test_pred = np.clip(np.exp(ols_model.predict(baseline_test_X).to_numpy()), 0, None)

display(pd.DataFrame([metrics_row("OLS baseline", test_df["demandKwh"], baseline_test_pred)]).round(4))

In [ ]:
def make_forest_matrix(frame):
    matrix = pd.DataFrame(
        {
            "pricePerKwh": frame["maxPricePerKwh"],
            "logPrice": frame["logPrice"],
            "hour": frame["hour"],
            "month": frame["month"],
            "dayOfWeek": frame["dayOfWeek"],
            "isWeekend": frame["isWeekend"],
            "hourSin": frame["hourSin"],
            "hourCos": frame["hourCos"],
            "monthSin": frame["monthSin"],
            "monthCos": frame["monthCos"],
            "lagDemand1": frame["lagDemand1"],
            "lagDemand24": frame["lagDemand24"],
            "lagPrice1": frame["lagPrice1"],
            "lagPrice24": frame["lagPrice24"],
            "rollingDemand24": frame["rollingDemand24"],
            "rollingPrice24": frame["rollingPrice24"],
            "logPrice_x_isWeekend": frame["logPrice"] * frame["isWeekend"],
            "logPrice_x_hourSin": frame["logPrice"] * frame["hourSin"],
            "logPrice_x_hourCos": frame["logPrice"] * frame["hourCos"],
            "lagDemandDelta": frame["lagDemand1"] - frame["lagDemand24"],
            "lagPriceDelta": frame["lagPrice1"] - frame["lagPrice24"],
            "rollingDemandDelta": frame["rollingDemand24"] - frame["lagDemand24"],
        }
    )
    return matrix


rf_train_df = train_df.tail(3200).copy()
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    random_state=20260427,
    n_jobs=-1,
)
rf_model.fit(make_forest_matrix(rf_train_df), rf_train_df["demandKwh"])

rf_validation_pred = np.clip(rf_model.predict(make_forest_matrix(validation_df)), 0, None)
rf_test_pred = np.clip(rf_model.predict(make_forest_matrix(test_df)), 0, None)
validation_residuals = validation_df["demandKwh"].to_numpy() - rf_validation_pred

metrics = pd.DataFrame(
    [
        metrics_row("OLS baseline", test_df["demandKwh"], baseline_test_pred),
        metrics_row("Random Forest", test_df["demandKwh"], rf_test_pred),
    ]
).round(4)

display(metrics)
display(pd.Series(validation_residuals).describe().to_frame("validation residuals").round(4))

In [ ]:
comparison = test_df[["hour_bucket", "demandKwh"]].copy()
comparison["ols_pred"] = baseline_test_pred
comparison["rf_pred"] = rf_test_pred

ax = comparison.tail(168).plot(
    x="hour_bucket",
    y=["demandKwh", "ols_pred", "rf_pred"],
    figsize=(12, 4),
    title="Last 168 test hours: actual vs predictions",
)
ax.set_ylabel("kWh")
plt.show()

## 3. Supply side: 从 solar generation 构造小时级供给参考

这里和前端 `solarSupplyReports.ts` 的生成规则一致：

- 先把正的 `SolarGeneration` 从 15 分钟累加到 `site-hour`
- 再对同一小时所有站点取中位数
- 用 `/ 4` 缩放成典型 household surplus
- 用 abundance 映射出一个参考挂牌价

In [ ]:
supply_chunks = []

for chunk in pd.read_csv(SUPPLY_CSV, usecols=["SiteKey", "Timestamp", "SolarGeneration"], chunksize=200_000):
    chunk["SolarGeneration"] = pd.to_numeric(chunk["SolarGeneration"], errors="coerce")
    chunk["Timestamp"] = pd.to_datetime(chunk["Timestamp"], utc=True, errors="coerce")
    chunk = chunk.dropna(subset=["SiteKey", "Timestamp", "SolarGeneration"]).copy()
    chunk = chunk.loc[chunk["SolarGeneration"] > 0].copy()

    if chunk.empty:
        continue

    chunk["hour_bucket"] = chunk["Timestamp"].dt.floor("h")
    site_hour = chunk.groupby(["SiteKey", "hour_bucket"], as_index=False)["SolarGeneration"].sum()
    supply_chunks.append(site_hour)

site_hour_supply = pd.concat(supply_chunks, ignore_index=True)
site_hour_supply = site_hour_supply.groupby(["SiteKey", "hour_bucket"], as_index=False)["SolarGeneration"].sum()


def aggregate_supply_hour(group):
    typical_surplus = percentile(group["SolarGeneration"].to_numpy(), 0.5) / HOUSEHOLD_SCALE_FACTOR
    return pd.Series(
        {
            "activeSiteCount": int(group["SiteKey"].nunique()),
            "surplusKwh": round(typical_surplus, 2),
        }
    )


hourly_supply = (
    site_hour_supply.groupby("hour_bucket", sort=True)
    .apply(aggregate_supply_hour)
    .reset_index()
    .sort_values("hour_bucket")
    .reset_index(drop=True)
)

surplus_p10 = percentile(hourly_supply["surplusKwh"], 0.1)
surplus_p90 = percentile(hourly_supply["surplusKwh"], 0.9)

abundance = (hourly_supply["surplusKwh"] - surplus_p10) / max(surplus_p90 - surplus_p10, 1e-9)
hourly_supply["normalizedAbundance"] = abundance.clip(0, 1)
hourly_supply["pricePerKwh"] = (0.19 - hourly_supply["normalizedAbundance"] * 0.07).clip(0.11, 0.19).round(3)

print(f"Supply rows: {len(hourly_supply):,}")
print("Date range:", hourly_supply["hour_bucket"].min(), "to", hourly_supply["hour_bucket"].max())
display(hourly_supply.head())

## 4. 用验证残差 + 小时级 supply 参考做定价

这一步对应前端里的 grid search pricing：

- 先取挂牌小时附近的 supply 历史样本  
- 估算 `FiT opportunity cost` 与 `retail shortfall penalty`  
- 对一组候选价格逐个模拟 expected profit  
- 选出净收益最高的价格

In [ ]:
def build_context(feature_table, listing_time):
    listing_ts = ensure_utc(listing_time)
    anchor_candidates = feature_table.loc[feature_table["hour_bucket"] <= listing_ts]
    anchor = anchor_candidates.iloc[-1] if not anchor_candidates.empty else feature_table.iloc[-1]

    previous_hour_match = feature_table.loc[feature_table["hour_bucket"] == listing_ts - pd.Timedelta(hours=1)]
    previous_day_match = feature_table.loc[feature_table["hour_bucket"] == listing_ts - pd.Timedelta(hours=24)]
    previous_hour = previous_hour_match.iloc[-1] if not previous_hour_match.empty else anchor
    previous_day = previous_day_match.iloc[-1] if not previous_day_match.empty else previous_hour

    hour_slice = feature_table.loc[feature_table["hour"] == listing_ts.hour]
    hour_default_demand = hour_slice["demandKwh"].mean() if not hour_slice.empty else feature_table["demandKwh"].mean()
    hour_default_price = hour_slice["maxPricePerKwh"].mean() if not hour_slice.empty else feature_table["maxPricePerKwh"].mean()

    return {
        "hour_bucket": listing_ts,
        "hour": listing_ts.hour,
        "month": listing_ts.month,
        "dayOfWeek": listing_ts.dayofweek,
        "isWeekend": int(listing_ts.dayofweek >= 5),
        "hourSin": np.sin(2 * np.pi * listing_ts.hour / 24),
        "hourCos": np.cos(2 * np.pi * listing_ts.hour / 24),
        "monthSin": np.sin(2 * np.pi * listing_ts.month / 12),
        "monthCos": np.cos(2 * np.pi * listing_ts.month / 12),
        "lagDemand1": float(previous_hour["demandKwh"]),
        "lagDemand24": float(previous_day["demandKwh"]),
        "lagPrice1": float(previous_hour["maxPricePerKwh"]),
        "lagPrice24": float(previous_day["maxPricePerKwh"]),
        "rollingDemand24": float(anchor["rollingDemand24"]) if pd.notna(anchor["rollingDemand24"]) else float(hour_default_demand),
        "rollingPrice24": float(anchor["rollingPrice24"]) if pd.notna(anchor["rollingPrice24"]) else float(hour_default_price),
    }


def context_frame(context, candidate_price):
    frame = pd.DataFrame(
        [
            {
                "maxPricePerKwh": candidate_price,
                "logPrice": np.log(max(candidate_price, 0.005)),
                "hour": context["hour"],
                "month": context["month"],
                "dayOfWeek": context["dayOfWeek"],
                "isWeekend": context["isWeekend"],
                "hourSin": context["hourSin"],
                "hourCos": context["hourCos"],
                "monthSin": context["monthSin"],
                "monthCos": context["monthCos"],
                "lagDemand1": context["lagDemand1"],
                "lagDemand24": context["lagDemand24"],
                "lagPrice1": context["lagPrice1"],
                "lagPrice24": context["lagPrice24"],
                "rollingDemand24": context["rollingDemand24"],
                "rollingPrice24": context["rollingPrice24"],
            }
        ]
    )
    return frame


def predict_baseline_demand(context, candidate_price):
    row = context_frame(context, candidate_price)
    design = make_baseline_design(row).reindex(columns=baseline_columns, fill_value=0.0)
    prediction = float(np.exp(ols_model.predict(design).iloc[0]))
    return max(0.0, prediction)


def predict_rf_demand(context, candidate_price):
    row = context_frame(context, candidate_price)
    prediction = float(rf_model.predict(make_forest_matrix(row))[0])
    return max(0.0, prediction)


def select_relevant_supply_rows(supply_table, listing_time, max_count=120):
    listing_ts = ensure_utc(listing_time)
    ranked = supply_table.copy()
    ranked["hourGap"] = ranked["hour_bucket"].dt.hour.map(lambda hour: circular_distance(hour, listing_ts.hour, 24))
    ranked["monthGap"] = ranked["hour_bucket"].dt.month.map(lambda month: circular_distance(month, listing_ts.month, 12))
    ranked = ranked.sort_values(["hourGap", "monthGap", "hour_bucket"], ascending=[True, True, False])
    return ranked.head(max_count).reset_index(drop=True)


def derive_supply_residuals(target_surplus_kwh, relevant_supply):
    if len(relevant_supply) < 4:
        return np.asarray([-0.3, -0.15, -0.05, 0.0, 0.05, 0.15, 0.3]) * target_surplus_kwh

    comparable = relevant_supply["surplusKwh"].to_numpy(dtype=float)
    median_surplus = percentile(comparable, 0.5)
    scale = target_surplus_kwh / max(median_surplus, 0.5)
    centered = (comparable - median_surplus) * scale
    centered = centered - centered.mean()
    centered = np.clip(centered, -target_surplus_kwh * 0.85, target_surplus_kwh * 0.85)
    return compress_samples(np.sort(centered), 40)


def price_responsive_share(candidate_price, fit_price, retail_tariff):
    if retail_tariff <= fit_price:
        return 0.95 if candidate_price <= fit_price else 0.05

    midpoint = (fit_price + retail_tariff) / 2
    steepness = 8 / (retail_tariff - fit_price)
    logistic_share = 1 / (1 + np.exp(steepness * (candidate_price - midpoint)))

    if candidate_price >= retail_tariff:
        return 0.0

    return clamp(float(logistic_share), 0.0, 0.995)


def simulate_price_outcome(
    candidate_price,
    context,
    target_surplus_kwh,
    fit_price,
    retail_tariff,
    demand_residual_samples,
    supply_residual_samples,
    historical_demand_range,
):
    gross_predicted_demand = clamp(
        predict_rf_demand(context, candidate_price),
        historical_demand_range[0] * 0.5,
        historical_demand_range[1] * 1.4,
    )
    market_share = price_responsive_share(candidate_price, fit_price, retail_tariff)

    total_profit = 0.0
    total_demand = 0.0
    total_shortfall = 0.0
    total_own_supply_used = 0.0
    total_sellthrough_hits = 0
    scenario_count = 0

    for demand_residual in demand_residual_samples:
        simulated_gross_demand = max(0.0, gross_predicted_demand + float(demand_residual))
        simulated_demand = simulated_gross_demand * market_share

        for supply_residual in supply_residual_samples:
            available_supply = max(0.0, target_surplus_kwh + float(supply_residual))
            own_supply_used = min(simulated_demand, available_supply)
            shortfall = max(0.0, simulated_demand - available_supply)
            profit = own_supply_used * (candidate_price - fit_price) - shortfall * max(retail_tariff - candidate_price, 0.0)

            total_profit += profit
            total_demand += simulated_demand
            total_shortfall += shortfall
            total_own_supply_used += own_supply_used
            total_sellthrough_hits += int(simulated_demand >= target_surplus_kwh)
            scenario_count += 1

    return {
        "expectedProfit": total_profit / max(scenario_count, 1),
        "expectedDemand": total_demand / max(scenario_count, 1),
        "expectedShortfall": total_shortfall / max(scenario_count, 1),
        "expectedOwnSupplyUsed": total_own_supply_used / max(scenario_count, 1),
        "fillProbability": total_sellthrough_hits / max(scenario_count, 1),
        "marketShare": market_share,
    }

In [ ]:
context = build_context(demand_features, LISTING_TIME)
relevant_supply = select_relevant_supply_rows(hourly_supply, LISTING_TIME)
supply_residuals = derive_supply_residuals(TARGET_SURPLUS_KWH, relevant_supply)
demand_residuals = compress_samples(validation_residuals, 160)

fit_price = round(clamp(percentile(relevant_supply["pricePerKwh"], 0.15), 0.02, 0.30), 3)
demand_p10 = percentile(hourly_demand["maxPricePerKwh"], 0.10)
demand_p35 = percentile(hourly_demand["maxPricePerKwh"], 0.35)
demand_p70 = percentile(hourly_demand["maxPricePerKwh"], 0.70)
demand_p85 = percentile(hourly_demand["maxPricePerKwh"], 0.85)
demand_p95 = percentile(hourly_demand["maxPricePerKwh"], 0.95)
retail_tariff = round(clamp(max(demand_p85, fit_price + 0.03), fit_price + 0.03, 0.45), 3)

price_floor = round(clamp(max(fit_price + 0.005, demand_p10 * 0.8), fit_price + 0.005, retail_tariff), 3)
price_ceiling = round(clamp(max(price_floor + 0.02, demand_p95 * 1.02), price_floor + 0.02, retail_tariff), 3)

price_grid = np.unique(np.round(np.linspace(price_floor, price_ceiling, 48), 3))
historical_demand_range = (
    max(0.01, float(demand_features["demandKwh"].min())),
    max(0.05, float(demand_features["demandKwh"].max())),
)

rows = []
for candidate_price in price_grid:
    outcome = simulate_price_outcome(
        candidate_price=candidate_price,
        context=context,
        target_surplus_kwh=TARGET_SURPLUS_KWH,
        fit_price=fit_price,
        retail_tariff=retail_tariff,
        demand_residual_samples=demand_residuals,
        supply_residual_samples=supply_residuals,
        historical_demand_range=historical_demand_range,
    )
    rows.append({"candidatePrice": candidate_price, **outcome})

pricing_grid = pd.DataFrame(rows)
optimized_row = pricing_grid.sort_values("expectedProfit", ascending=False).iloc[0]
current_outcome = simulate_price_outcome(
    candidate_price=TARGET_PRICE,
    context=context,
    target_surplus_kwh=TARGET_SURPLUS_KWH,
    fit_price=fit_price,
    retail_tariff=retail_tariff,
    demand_residual_samples=demand_residuals,
    supply_residual_samples=supply_residuals,
    historical_demand_range=historical_demand_range,
)

summary = pd.DataFrame(
    [
        {
            "listingTime": LISTING_TIME,
            "targetPrice": TARGET_PRICE,
            "optimizedPrice": optimized_row["candidatePrice"],
            "fitPrice": fit_price,
            "retailTariff": retail_tariff,
            "currentExpectedProfit": current_outcome["expectedProfit"],
            "optimizedExpectedProfit": optimized_row["expectedProfit"],
            "expectedDemandAtOptimum": optimized_row["expectedDemand"],
            "expectedShortfallAtOptimum": optimized_row["expectedShortfall"],
            "baselineDemandAtOptimum": predict_baseline_demand(context, optimized_row["candidatePrice"]),
            "randomForestDemandAtOptimum": predict_rf_demand(context, optimized_row["candidatePrice"]),
            "referenceSupplyRows": len(relevant_supply),
            "demandResidualCount": len(demand_residuals),
            "supplyResidualCount": len(supply_residuals),
        }
    ]
).round(4)

display(summary)
display(pricing_grid.sort_values("expectedProfit", ascending=False).head(10).round(4))

In [ ]:
ax = pricing_grid.plot(
    x="candidatePrice",
    y="expectedProfit",
    figsize=(8, 4),
    title="Grid search over listing prices",
    legend=False,
)
ax.axvline(TARGET_PRICE, color="tab:orange", linestyle="--", label="target price")
ax.axvline(float(optimized_row["candidatePrice"]), color="tab:green", linestyle="--", label="optimized price")
ax.set_ylabel("expected profit")
ax.legend()
plt.show()

## 5. 你可以怎么改这个 notebook

- 想和前端完全同一个输入样本：直接改 `TARGET_SURPLUS_KWH`、`TARGET_PRICE`、`LISTING_TIME`
- 想换 demand 数据：改 `DEMAND_CSV`
- 想换 supply 数据：改 `SUPPLY_CSV`
- 想把结果写回前端默认数据文件：继续用仓库里的 `generate-demand-side.mjs` 和 `generate-supply-side.mjs`

如果你后面把前端模型再改了，这个 notebook 也可以通过 `scripts/generate-pricing-notebook.py` 重新生成。